# 06 — Optimizing prompts with GEPA

**Reflective prompt evolution against a small gold set.**

Orqest already produces every signal an optimizer needs (accuracy via your scoring function, confidence via `EnrichedOutput`, latency via `Tracer.Span`, cost via `pydantic_ai.usage.RunUsage`). This notebook closes the loop: a separate *reflection LLM* reads execution traces and proposes prompt mutations, GEPA's Pareto frontier prevents collapse into local optima, and we watch a small `ResearchSummarizerAgent` evolve from baseline to optimized in ~5–10 minutes.

**Prerequisites:**
- A `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL`.
- The optimization extra installed: `uv sync --group optimization`.
- ~$1–3 of LLM budget if running end-to-end (smaller if you cap `max_metric_calls`).

## 1. Load config + define the agent

A small typed agent that summarizes a paragraph in response to a question. The output schema includes a `confidence_self` field so `StructuredOutputProtocol` has something to read.

In [7]:
from typing import Any

from pydantic import BaseModel, Field

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState

config = load_config()
print(f"Task model: {config.llm_model}")

class Summary(BaseModel):
    answer: str = Field(description="The direct answer, or 'the paragraph does not say'.")
    key_points: list[str] = Field(default_factory=list, max_length=3)
    self_confidence: float = Field(ge=0.0, le=1.0, description="Your confidence in this answer.")

INITIAL_PROMPT = (
    "You are a research summarizer. Read the paragraph and answer the question concisely."
)

class ResearchSummarizerAgent(BaseAgent[GlobalState, Summary]):
    async def _run_implementation(self, state: GlobalState, **kwargs: Any) -> Summary:
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output

Task model: openai:gpt-5.2


## 2. The 15-example gold set

Three buckets of 5:
- **Factual** — answer is in the paragraph; tests anti-hallucination.
- **Inferential** — answer requires combining facts; tests reasoning.
- **Adversarial** — paragraph doesn't actually contain the answer; correct response is to abstain.

Hand-writing examples is the hardest part of optimization. W2 will ship `synthetic_gold.py` to bootstrap an eval set from a strong model — for now we do it the honest way.

In [8]:
from orqest.optimization import GoldExample

class Question(BaseModel):
    paragraph: str
    question: str
    bucket: str  # factual | inferential | adversarial

def _ex(paragraph: str, question: str, expected: str, bucket: str) -> GoldExample:
    return GoldExample[Question, Summary](
        input=Question(paragraph=paragraph, question=question, bucket=bucket),
        expected=Summary(answer=expected, self_confidence=0.9),
        id=f"{bucket}-{hash(paragraph) % 10000}",
    )

ABSTAIN = "the paragraph does not say"

EXAMPLES = [
    # Factual
    _ex("The Eiffel Tower was completed in 1889 for the World's Fair.", "When was the Eiffel Tower completed?", "1889", "factual"),
    _ex("Photosynthesis converts CO2 and water into glucose using sunlight.", "What does photosynthesis convert CO2 into?", "glucose", "factual"),
    _ex("Mount Everest stands at 8,849 meters above sea level.", "How tall is Mount Everest?", "8,849 meters", "factual"),
    _ex("Python was created by Guido van Rossum and first released in 1991.", "Who created Python?", "Guido van Rossum", "factual"),
    _ex("The human heart pumps about 7,000 liters of blood per day.", "How much blood does the heart pump daily?", "7,000 liters", "factual"),
    # Inferential — need to combine facts
    _ex("A factory ran 8 hours yesterday at 100 units/hour. Today it ran 6 hours at 120 units/hour.", "Did the factory produce more today or yesterday?", "yesterday", "inferential"),
    _ex("Alice is taller than Bob. Bob is taller than Carol. Carol is 5'4\".", "Who is the tallest?", "Alice", "inferential"),
    _ex("The library closes at 9pm on weekdays and 6pm on weekends. Today is Saturday.", "What time does the library close today?", "6pm", "inferential"),
    _ex("Train A leaves at 3pm taking 2 hours. Train B leaves at 4pm taking 30 minutes.", "Which train arrives first?", "Train B", "inferential"),
    _ex("Sales were $100k in Q1, doubled in Q2, halved in Q3, then tripled in Q4.", "What were Q4 sales?", "$300k", "inferential"),
    # Adversarial — answer is NOT in the paragraph
    _ex("The Atlantic Ocean separates the Americas from Europe and Africa.", "How deep is the Pacific Ocean?", ABSTAIN, "adversarial"),
    _ex("Albert Einstein developed the theory of general relativity in 1915.", "What is Einstein's birthday?", ABSTAIN, "adversarial"),
    _ex("Pizza originated in Naples, Italy.", "What is the population of Naples?", ABSTAIN, "adversarial"),
    _ex("The novel 1984 was written by George Orwell.", "How many copies of 1984 have been sold?", ABSTAIN, "adversarial"),
    _ex("Coffee contains caffeine, a natural stimulant.", "What is the chemical formula of caffeine?", ABSTAIN, "adversarial"),
]
print(f"Loaded {len(EXAMPLES)} gold examples across 3 buckets.")

Loaded 15 gold examples across 3 buckets.


## 3. Wrap the agent in an `Evaluator`

The `agent_factory` returns a *fresh* agent for each evaluation (mutating an existing one is unsafe — the cached `pydantic_ai.Agent` keeps the old prompt). The `score_fn` is intentionally simple: substring match for factual/inferential, abstention check for adversarial.

In [9]:
from orqest.optimization import Evaluator

def make_agent(decoded: dict[str, Any]) -> ResearchSummarizerAgent:
    return ResearchSummarizerAgent(
        agent_name="research_summarizer",
        system_prompt=decoded["researcher.system_prompt"],
        output_type=Summary,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )

def score(output: Summary, ex: GoldExample[Question, Summary]) -> float:
    expected = (ex.expected.answer if ex.expected else "").lower()
    actual = output.answer.lower()
    if ex.input.bucket == "adversarial":
        # Reward abstention; penalize fabrication.
        return 1.0 if ("does not say" in actual or "cannot" in actual or "unknown" in actual) else 0.0
    return 1.0 if expected in actual else 0.0

# Wrap the question into the prompt the agent receives
class QuestionEvaluator(Evaluator[Question, Summary]):
    async def _run_agent(self, agent, example):
        state = GlobalState()
        prompt = f"Paragraph: {example.input.paragraph}\n\nQuestion: {example.input.question}"
        return await agent.call_model(prompt, state)

evaluator = QuestionEvaluator(agent_factory=make_agent, score_fn=score, confidence_protocol=object())
print("Evaluator ready.")

Evaluator ready.


## 4. Baseline — measure before optimizing

In [10]:
import statistics
from collections import defaultdict

async def measure(decoded: dict[str, Any], examples: list[GoldExample]) -> dict[str, float]:
    bundles = await evaluator.evaluate_batch(decoded, examples)
    by_bucket: dict[str, list[float]] = defaultdict(list)
    for ex, b in zip(examples, bundles, strict=True):
        by_bucket[ex.input.bucket].append(b.accuracy)
    return {bucket: statistics.mean(scores) for bucket, scores in by_bucket.items()} | {
        "overall": statistics.mean(b.accuracy for b in bundles),
        "mean_latency_ms": statistics.mean(b.latency_ms for b in bundles),
    }

baseline = await measure({"researcher.system_prompt": INITIAL_PROMPT}, EXAMPLES)
print("Baseline:")
for k, v in baseline.items():
    print(f"  {k:20s} {v:.3f}")

Baseline:
  factual              1.000
  inferential          0.800
  adversarial          1.000
  overall              0.933
  mean_latency_ms      1929.389


## 5. Define the genome

One `PromptGene` for the system prompt. The `constraints` text is surfaced verbatim to the reflection LLM — it's a powerful lever for nudging behaviour without touching the prompt itself.

In [11]:
from orqest.optimization import Genome, PromptGene

genome = Genome(genes=[
    PromptGene(
        name="researcher.system_prompt",
        initial=INITIAL_PROMPT,
        constraints=(
            "You MUST abstain (say 'the paragraph does not say') when the paragraph "
            "does not contain the answer. Never invent facts not in the paragraph."
        ),
    ),
])
print(f"Genome: {len(genome.genes)} gene(s); seed = {genome.to_seed()}")

Genome: 1 gene(s); seed = {'researcher.system_prompt': 'You are a research summarizer. Read the paragraph and answer the question concisely.'}


## 6. Run the optimizer

**~5–10 minutes; ~$1–$3 in LLM cost.** The reflection model is the optimizer's brain — use the strongest model you can afford for it. The task model can stay cheap.

In [ ]:
from orqest.optimization import OptimizationConfig, OptimizationRunner

opt_config = OptimizationConfig(
    max_metric_calls=80,                            # keep small for the demo
    reflection_model=config.llm_model,              # ideally a stronger model
    task_model=config.llm_model,
    minibatch_size=3,
    valset_fraction=0.4,                            # 6 train / 9 val
    seed=42,
)

runner = OptimizationRunner(opt_config, genome=genome, evaluator=evaluator)
result = await runner.optimize(EXAMPLES)
print(f"Best aggregate score: {result.best_score:.3f}")
print(f"Pareto frontier size: {len(result.pareto_candidates)}")

ModuleNotFoundError: No module named 'litellm'

## 7. Before / after diff

In [ ]:
from orqest.optimization import apply_result

# Build a fresh baseline agent against which to diff
baseline_agent = make_agent({"researcher.system_prompt": INITIAL_PROMPT})
diffs = apply_result(result, target=baseline_agent)  # dry_run=True (default)
for d in diffs:
    if d.changed:
        print(d.unified)

## 8. Re-measure with the evolved prompt

In [ ]:
evolved = await measure(result.best_decoded, EXAMPLES)
print("Per-bucket accuracy:")
print(f"  {'bucket':<14s} {'before':>10s} {'after':>10s} {'delta':>10s}")
for bucket in ("factual", "inferential", "adversarial", "overall"):
    b, a = baseline.get(bucket, 0.0), evolved.get(bucket, 0.0)
    arrow = "↑" if a > b else ("↓" if a < b else "=")
    print(f"  {bucket:<14s} {b:>10.3f} {a:>10.3f}     {a - b:+.3f} {arrow}")

## 9. The Pareto frontier

`result.pareto_candidates` is the **real** output. The aggregate winner is one slice; other Pareto candidates may win on *some* example or *some* objective dimension that the aggregate scalar discounts.

In [ ]:
for i, cand in enumerate(result.pareto_candidates):
    prompt = cand.get("researcher.system_prompt", "")
    print(f"--- Pareto candidate {i + 1} ---")
    print(prompt[:300] + ("..." if len(prompt) > 300 else ""))
    print()

## 10. Commit the winner (opt-in)

`apply_result(dry_run=False)` mutates the agent's `system_prompt` AND invalidates the cached `pydantic_ai.Agent` — without that cache reset the new prompt would be silently ignored at runtime. Always opt in explicitly; never auto-commit.

In [ ]:
production_agent = make_agent({"researcher.system_prompt": INITIAL_PROMPT})
diffs = apply_result(result, target=production_agent, dry_run=False)
print(f"Committed {sum(1 for d in diffs if d.changed)} change(s).")
print(f"New system_prompt:\n{production_agent.system_prompt}")

## What's next

- **W2 — synthetic gold.** Hand-writing 15 examples is the hardest part of any optimization. The next wave ships a `synthetic_gold.py` module that bootstraps an eval set from a strong model.
- **Notebook 07 — compound optimization.** Optimizes the planner inside `MetaOrchestrator` and watches the entire orchestration improve downstream — the same machinery, applied to a more compound surface.
- See `docs/concepts/optimization.md` for the full architecture and gotcha list.